### Ridge, Lasso and Elastic Net
In order for Ridge and Lasso (and Elastic net) to have an effect, you must use scaled data to build the models, since regularization depends on coefficient magnitude, and if using non-scaled data the penalty will affect them unequally. Feel free to use this code to scale the data:

```python
# Standardize X
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# Standardize y
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).flatten()
```
For this task you must do the following:
   - Ridge regression (using multiple alphas)
   - Lasso regression (using multiple alphas)
   - Elastic Net (using multiple alphas)
 - Discussion and conclusion:
   - Discuss the MSE and $R^2$ of all 3 models and conclude which model has the best performance - note the MSE will be scaled!
   - Rebuild the OLS model from Task 4, but this time use the scaled data from this task - interpret the meaning of the model's coefficients
   - Use the coefficients of the best ridge and lasso model to print the 5 most important features and compare to the 5 most important features in the OLS with scaled data model. Do the models agree about which features are the most important?

Note: You may get a convergence warning; try increasing the `max_iter` parameter of the model (the default is 1000 - maybe set it to 100000)

In [ ]:
# Use this for Ridge, Lasso and Elastic Net. Add more code blocks if needed.

from sklearn.preprocessing import StandardScaler
import numpy as np

# Standardize X
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# Standardize y
scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.to_numpy().reshape(-1, 1)).ravel()
y_test_scaled = scaler_y.transform(y_test.to_numpy().reshape(-1, 1)).ravel()

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd

alphas = [0.001, 0.01, 0.1, 1, 10, 100]

results = []

# Ridge
for a in alphas:
    model = Ridge(alpha=a)
    model.fit(X_train_scaled, y_train_scaled)
    pred = model.predict(X_test_scaled)
    results.append(("Ridge", a, mean_squared_error(y_test_scaled, pred), r2_score(y_test_scaled, pred)))

# Lasso
for a in alphas:
    model = Lasso(alpha=a, max_iter=100000)
    model.fit(X_train_scaled, y_train_scaled)
    pred = model.predict(X_test_scaled)
    results.append(("Lasso", a, mean_squared_error(y_test_scaled, pred), r2_score(y_test_scaled, pred)))

# Elastic Net
for a in alphas:
    model = ElasticNet(alpha=a, l1_ratio=0.5, max_iter=100000)
    model.fit(X_train_scaled, y_train_scaled)
    pred = model.predict(X_test_scaled)
    results.append(("ElasticNet", a, mean_squared_error(y_test_scaled, pred), r2_score(y_test_scaled, pred)))

results_df = pd.DataFrame(results, columns=["Model", "Alpha", "MSE_scaled", "R2"])
results_df.sort_values(["Model", "R2"], ascending=[True, False])

,Model,Alpha,MSE_scaled,R2
13,ElasticNet,0.010,0.123783,0.865947
12,ElasticNet,0.001,0.124974,0.864658
14,ElasticNet,0.100,0.135476,0.853285
15,ElasticNet,1.000,0.566144,0.386887
16,ElasticNet,10.000,0.924087,-0.000753
17,ElasticNet,100.000,0.924087,-0.000753
7,Lasso,0.010,0.122946,0.866854
6,Lasso,0.001,0.124736,0.864916
8,Lasso,0.100,0.148866,0.838784
9,Lasso,1.000,0.924087,-0.000753


In [ ]:
best_per_model = results_df.sort_values("R2", ascending=False).groupby("Model").head(1)
best_per_model

,Model,Alpha,MSE_scaled,R2
7,Lasso,0.010,0.122946,0.866854
13,ElasticNet,0.010,0.123783,0.865947
0,Ridge,0.001,0.125188,0.864426


In [ ]:
best_overall = results_df.sort_values("R2", ascending=False).iloc[0]
best_overall

Model            Lasso
Alpha             0.01
MSE_scaled    0.122946
R2            0.866854
Name: 7, dtype: object

In [ ]:
from sklearn.linear_model import LinearRegression

ols_scaled = LinearRegression()
ols_scaled.fit(X_train_scaled, y_train_scaled)

pred_ols = ols_scaled.predict(X_test_scaled)
mse_ols = mean_squared_error(y_test_scaled, pred_ols)
r2_ols = r2_score(y_test_scaled, pred_ols)

mse_ols, r2_ols

(0.12518756276373944, 0.8644264443647276)

In [ ]:
import numpy as np

feature_names = X.columns.tolist()

def top_features(coefs, feature_names, top_n=5):
    coefs = np.array(coefs).ravel()
    idx = np.argsort(np.abs(coefs))[::-1][:top_n]
    return [(feature_names[i], coefs[i]) for i in idx]

In [ ]:
ols_top5 = top_features(ols_scaled.coef_, feature_names, 5)
ols_top5

[('Original Price (DKK)', np.float64(0.8500387987374612)),
 ('Model Year', np.float64(0.16797776925233643)),
 ('Mileage (km)', np.float64(-0.10321027416293946)),
 ('0-100 km/h (s)', np.float64(0.07702222554767721)),
 ('Electric Range (km)', np.float64(0.0706063294585927))]

In [ ]:
best_ridge_alpha = best_per_model[best_per_model["Model"]=="Ridge"]["Alpha"].values[0]
best_lasso_alpha = best_per_model[best_per_model["Model"]=="Lasso"]["Alpha"].values[0]

ridge_best = Ridge(alpha=best_ridge_alpha).fit(X_train_scaled, y_train_scaled)
lasso_best = Lasso(alpha=best_lasso_alpha, max_iter=100000).fit(X_train_scaled, y_train_scaled)

ridge_top5 = top_features(ridge_best.coef_, feature_names, 5)
lasso_top5 = top_features(lasso_best.coef_, feature_names, 5)

ols_top5, ridge_top5, lasso_top5

([('Original Price (DKK)', np.float64(0.8500387987374612)),
  ('Model Year', np.float64(0.16797776925233643)),
  ('Mileage (km)', np.float64(-0.10321027416293946)),
  ('0-100 km/h (s)', np.float64(0.07702222554767721)),
  ('Electric Range (km)', np.float64(0.0706063294585927))],
 [('Original Price (DKK)', np.float64(0.8500383098066201)),
  ('Model Year', np.float64(0.16797769574899796)),
  ('Mileage (km)', np.float64(-0.10321030521488687)),
  ('0-100 km/h (s)', np.float64(0.07702217148882606)),
  ('Electric Range (km)', np.float64(0.07060635987811895))],
 [('Original Price (DKK)', np.float64(0.84190404858542)),
  ('Model Year', np.float64(0.1691259618274574)),
  ('Mileage (km)', np.float64(-0.0948309229887194)),
  ('Electric Range (km)', np.float64(0.06339273848104761)),
  ('Annual Road Tax (DKK)', np.float64(-0.05117838611369573))])